### Q1: Embedding a query

Used embedder.py to create embedder model and encode query into vector

In [81]:

from embedder import Embedder

In [4]:
model = Embedder()

query = "How does approximate nearest neighbor search work?"

vector = model.encode(query)
len(vector)


384

In [3]:
vector[0]

np.float64(-0.02058203437252893)

### Q2: Find cosine similarity of encoded lessons with query.

First download documents

In [83]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Loop over documents to find specific filename

In [24]:
for file in documents:
    if file['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        chosen = (file['content'])
        break
    chosen = None


Encode the found documents content into a vector and find the dot product of that vector with the query vector

In [25]:
page_vector = model.encode(chosen)
vector.dot(page_vector)

np.float64(0.36107027225589694)

### Q3: Chunking and search by hand  

First import the chunks

In [26]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [29]:
len(chunks)

295

Create list of texts using content field of chunks

In [30]:
texts = []

for chunk in chunks:
    texts.append(chunk['content'])

len(texts)

295

In [31]:
texts[0]

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

Encode the texts into a matrix

In [32]:
X = model.encode_batch(texts)

In [33]:
X.shape

(295, 384)

Find the dot product of the vector and the matrix

In [34]:
scores = X.dot(vector)

Find the index of the maximum score and the chunk that index refers to

In [39]:
import numpy as np

idx = np.argmax(scores)
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

### Q4: Vector search with minsearch  

Fit an index to the chunks using the matrix found already

In [84]:
from minsearch import VectorSearch

In [50]:
index_vector = VectorSearch()
index_vector.fit(X, chunks)


Encode the query into a vector and search for that vector in the index. Print the filename of the first result. 

In [51]:
query = "What metric do we use to evaluate a search engine?"
query_vector = model.encode(query)

output = index_vector.search(query_vector,
             num_results=5,
             )
print(output[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


### Q5: Text search vs vector search

Create a text index using the chunks and fit it.

In [ ]:
from minsearch import Index

index_text = Index(text_fields=['content'])

index_text.fit(chunks)

Encode the query into a vector. Use query and vector to search the text and vector indices respectably.  
Print the results out.

In [64]:
query = "How do I store vectors in PostgreSQL?"

query_vector = model.encode(query)

output_vector = index_vector.search(query_vector, num_results=5)
output_text = index_text.search(query, num_results=5)

print('\n'.join([res['filename'] for res in output_text]))
print('')
print('\n'.join([res['filename'] for res in output_vector]))

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


### Q6: Hybrid search

Function that defined reciprocal rank fusion (RRF)

In [67]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Encode query and feed vector and query into indices to get results.  
Then calculate the RRF for both outputs together and print results.  

In [80]:
query = "How do I give the model access to tools?"

query_vector = model.encode(query)

output_vector = index_vector.search(query_vector, num_results=50)
output_text = index_text.search(query, num_results=50)

results = rrf([output_vector, output_text])
print('\n'.join([res['filename'] for res in results]))

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/01-intro.md
